In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import shutil, os
if not os.path.exists('/content/60_pic'):
    shutil.copytree('/content/drive/MyDrive/60_pic', '/content/60_pic')

In [3]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import pickle


In [ ]:
train_dir = '/content/60_pic'
img_size = 224
batch_size = 16

p1_epochs = 30
p1_lr = 3e-4


In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    brightness_range=[0.7, 1.3],
    zoom_range=0.18,
    horizontal_flip=True,
    width_shift_range=0.12,
    height_shift_range=0.12,
    shear_range=0.10,
    channel_shift_range=20.0,
    fill_mode='reflect',
    validation_split=0.2)

validation_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True)

validation_generator = validation_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False)

Found 1570 images belonging to 31 classes.
Found 387 images belonging to 31 classes.


In [ ]:
num_classes = train_generator.num_classes
print(f'Classes: {num_classes}, Train: {train_generator.n}, Val: {validation_generator.n}')

with open('/content/drive/MyDrive/class_names.pkl', 'wb') as f:
    pickle.dump(train_generator.class_indices, f)

cw = compute_class_weight('balanced', classes=np.unique(train_generator.classes), y=train_generator.classes)
class_weights = dict(enumerate(cw))

Classes: 31, Train: 1570, Val: 387


In [8]:
base = MobileNetV2(weights='imagenet',
                   include_top=False,
                   input_shape=(img_size, img_size, 3))
base.trainable = False

x = base.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, kernel_regularizer=regularizers.l2(1e-4), use_bias=False)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, kernel_regularizer=regularizers.l2(1e-4), use_bias=False)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.2)(x)
output = Dense(num_classes, activation='softmax')(x)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model = Model(inputs=base.input, outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(p1_lr),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,990,559 (11.41 MB)

 Trainable params: 728,735 (2.78 MB)

 Non-trainable params: 2,261,824 (8.63 MB)

In [ ]:
callbacks = [
    ModelCheckpoint('/content/drive/MyDrive/best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
]

history1 = model.fit(train_generator, validation_data=validation_generator,
                     epochs=p1_epochs, callbacks=callbacks, class_weight=class_weights)

Epoch 1/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1114 - loss: 3.6085
Epoch 1: val_accuracy improved from None to 0.48062, saving model to /content/drive/MyDrive/best_model.h5



Epoch 1: finished saving model to /content/drive/MyDrive/best_model.h5
99/99 ━━━━━━━━━━━━━━━━━━━━ 170s 1s/step - accuracy: 0.2032 - loss: 3.2422 - val_accuracy: 0.4806 - val_loss: 2.7586 - learning_rate: 3.0000e-04
Epoch 2/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 905ms/step - accuracy: 0.4961 - loss: 2.4053
Epoch 2: val_accuracy improved from 0.48062 to 0.61240, saving model to /content/drive/MyDrive/best_model.h5



Epoch 2: finished saving model to /content/drive/MyDrive/best_model.h5
99/99 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.5261 - loss: 2.2836 - val_accuracy: 0.6124 - val_loss: 2.2202 - learning_rate: 3.0000e-04
Epoch 3/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 866ms/step - accuracy: 0.6365 - loss: 1.9814
Epoch 3: val_accuracy improved from 0.61240 to 0.64341, saving model to /content/drive/MyDrive/best_model.h5



Epoch 3: finished saving model to /content/drive/MyDrive/best_model.h5
99/99 ━━━━━━━━━━━━━━━━━━━━ 105s 1s/step - accuracy: 0.6732 - loss: 1.9116 - val_accuracy: 0.6434 - val_loss: 1.9474 - learning_rate: 3.0000e-04
Epoch 4/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 876ms/step - accuracy: 0.7615 - loss: 1.6732
Epoch 4: val_accuracy improved from 0.64341 to 0.66925, saving model to /content/drive/MyDrive/best_model.h5



Epoch 4: finished saving model to /content/drive/MyDrive/best_model.h5
99/99 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.7643 - loss: 1.6509 - val_accuracy: 0.6693 - val_loss: 1.8071 - learning_rate: 3.0000e-04
Epoch 5/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 868ms/step - accuracy: 0.8035 - loss: 1.5359
Epoch 5: val_accuracy improved from 0.66925 to 0.70284, saving model to /content/drive/MyDrive/best_model.h5


...


Epoch 15: finished saving model to /content/drive/MyDrive/best_model.h5
99/99 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.9197 - loss: 1.1159 - val_accuracy: 0.8010 - val_loss: 1.4083 - learning_rate: 3.0000e-04
Epoch 16/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 930ms/step - accuracy: 0.9345 - loss: 1.0865
Epoch 16: val_accuracy improved from 0.80103 to 0.81137, saving model to /content/drive/MyDrive/best_model.h5



Epoch 16: finished saving model to /content/drive/MyDrive/best_model.h5
99/99 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.9268 - loss: 1.0954 - val_accuracy: 0.8114 - val_loss: 1.3985 - learning_rate: 3.0000e-04
Epoch 17/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 862ms/step - accuracy: 0.9381 - loss: 1.0834
Epoch 17: val_accuracy did not improve from 0.81137
99/99 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.9299 - loss: 1.1023 - val_accuracy: 0.7907 - val_loss: 1.4121 - learning_rate: 3.0000e-04
Epoch 18/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 850ms/step - accuracy: 0.9363 - loss: 1.0731
Epoch 18: val_accuracy did not improve from 0.81137
99/99 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.9395 - loss: 1.0716 - val_accuracy: 0.8036 - val_loss: 1.3891 - learning_rate: 3.0000e-04
Epoch 19/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 859ms/step - accuracy: 0.9361 - loss: 1.0685
Epoch 19: val_accuracy improved from 0.81137 to 0.81654, saving model to /content/drive/MyDrive/best_model.h5



Epoch 19: finished saving model to /content/drive/MyDrive/best_model.h5
99/99 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - accuracy: 0.9369 - loss: 1.0652 - val_accuracy: 0.8165 - val_loss: 1.4026 - learning_rate: 3.0000e-04
Epoch 20/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 854ms/step - accuracy: 0.9373 - loss: 1.0588
Epoch 20: val_accuracy did not improve from 0.81654
99/99 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.9357 - loss: 1.0703 - val_accuracy: 0.8088 - val_loss: 1.3947 - learning_rate: 3.0000e-04
Epoch 21/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 852ms/step - accuracy: 0.9429 - loss: 1.0606
Epoch 21: val_accuracy did not improve from 0.81654
99/99 ━━━━━━━━━━━━━━━━━━━━ 105s 1s/step - accuracy: 0.9382 - loss: 1.0525 - val_accuracy: 0.8140 - val_loss: 1.3753 - learning_rate: 3.0000e-04
Epoch 22/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 846ms/step - accuracy: 0.9603 - loss: 1.0267
Epoch 22: val_accuracy improved from 0.81654 to 0.81912, saving model to /content/drive/MyDrive/best_model.h5



Epoch 22: finished saving model to /content/drive/MyDrive/best_model.h5
99/99 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.9567 - loss: 1.0325 - val_accuracy: 0.8191 - val_loss: 1.3676 - learning_rate: 3.0000e-04
Epoch 23/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 840ms/step - accuracy: 0.9484 - loss: 1.0507
Epoch 23: val_accuracy did not improve from 0.81912
99/99 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.9548 - loss: 1.0415 - val_accuracy: 0.8114 - val_loss: 1.3717 - learning_rate: 3.0000e-04
Epoch 24/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 842ms/step - accuracy: 0.9429 - loss: 1.0619
Epoch 24: val_accuracy improved from 0.81912 to 0.82687, saving model to /content/drive/MyDrive/best_model.h5



Epoch 24: finished saving model to /content/drive/MyDrive/best_model.h5
99/99 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.9484 - loss: 1.0372 - val_accuracy: 0.8269 - val_loss: 1.3591 - learning_rate: 3.0000e-04
Epoch 25/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 844ms/step - accuracy: 0.9499 - loss: 1.0226
Epoch 25: val_accuracy did not improve from 0.82687
99/99 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.9452 - loss: 1.0351 - val_accuracy: 0.8217 - val_loss: 1.3427 - learning_rate: 3.0000e-04
Epoch 26/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 857ms/step - accuracy: 0.9547 - loss: 1.0293
Epoch 26: val_accuracy did not improve from 0.82687
99/99 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.9561 - loss: 1.0169 - val_accuracy: 0.8114 - val_loss: 1.3513 - learning_rate: 3.0000e-04
Epoch 27/30
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 852ms/step - accuracy: 0.9547 - loss: 1.0095
Epoch 27: val_accuracy did not improve from 0.82687
99/99 ━━━━━━━━━━━━━━━━━━━━ 105s 1s/step - accuracy: 0.9567 - loss: 1.0182 - va

In [ ]:
val_loss, val_acc = model.evaluate(validation_generator)
print(f'Final Val Accuracy: {val_acc:.4f}')

25/25 ━━━━━━━━━━━━━━━━━━━━ 17s 680ms/step - accuracy: 0.8398 - loss: 1.3185
Final Val Accuracy: 0.8398
